In [1]:
from torch.utils.data import DataLoader, random_split
from utils.report import AverageMeter
from utils.metrics import calculate_accuracy

import os
import random
import torch
import warnings

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


warnings.filterwarnings('ignore')

In [2]:
BATCH_SIZE = 128
LR = 0.0002
BETA1 = 0.5
BETA2 = 0.999
ES_THRES = 5
EPOCHS = 100
SEED = 1234
GAMMA = 10
PRETRAIN_DIR = "../dann/state_dicts/SVHN"

# Set Seed

In [3]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_everything(SEED)

In [4]:
from torchvision.datasets import MNIST, SVHN
import torchvision.transforms as transforms

In [5]:
download_root_mnist = '../datasets/MNIST_DATASET'
train_data_mnist = MNIST(download_root_mnist, train=True, download=True)
mean_mnist = train_data_mnist.data.float().mean() / 255
std_mnist = train_data_mnist.data.float().std() / 255

print(f'Calculated mean: {mean_mnist}')
print(f'Calculated std: {std_mnist}')

Calculated mean: 0.13066047430038452
Calculated std: 0.30810779333114624


In [6]:
train_transforms_mnist = transforms.Compose([
                            transforms.Grayscale(num_output_channels=3),
                            transforms.Resize(32),
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_mnist], std = [std_mnist])])

test_transforms_mnist = transforms.Compose([
                            transforms.Grayscale(num_output_channels=3),
                            transforms.Resize(32),
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_mnist], std = [std_mnist])])

In [7]:
train_valid_dataset_mnist = MNIST(download_root_mnist, transform=train_transforms_mnist, train=True, download=True)
train_dataset_mnist, valid_dataset_mnist = random_split(train_valid_dataset_mnist, [54000, 6000])
test_dataset_mnist = MNIST(download_root_mnist, transform=test_transforms_mnist, train=False, download=True)

In [8]:
download_root_svhn = '../datasets/SVHN_DATASET'
train_data_svhn = SVHN(download_root_svhn, split="train", download=True)
mean_svhn = train_data_svhn.data.mean() / 255
std_svhn = train_data_svhn.data.std() / 255

print(f'Calculated mean: {mean_svhn}')
print(f'Calculated std: {std_svhn}')

Using downloaded and verified file: ../datasets/SVHN_DATASET/train_32x32.mat
Calculated mean: 0.45141874380092256
Calculated std: 0.19929124669110937


In [9]:
train_transforms_svhn = transforms.Compose([
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_svhn], std = [std_svhn])])

test_transforms_svhn = transforms.Compose([
                            transforms.ToTensor(),
                            transforms.Normalize(mean = [mean_svhn], std = [std_svhn])])

In [10]:
train_valid_dataset_svhn = SVHN(download_root_svhn, transform=train_transforms_svhn, split="train", download=True)
train_dataset_svhn, valid_dataset_svhn = random_split(train_valid_dataset_svhn, [65931, 7326])
test_dataset_svhn = SVHN(download_root_svhn, transform=test_transforms_svhn, split="test", download=True)

Using downloaded and verified file: ../datasets/SVHN_DATASET/train_32x32.mat
Using downloaded and verified file: ../datasets/SVHN_DATASET/test_32x32.mat


# Model

In [11]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channel_dim=3, out_channel_dim=16):
        super(FeatureExtractor, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=in_channel_dim, out_channels=6, kernel_size=5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=out_channel_dim, kernel_size=5)
        
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        
#         x = x.view(-1, 16 * 5 * 5)

        return x

In [12]:
class Classifier(nn.Module):
    def __init__(self, in_dim, out_dim=10):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(in_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        
        return x

In [13]:
class Discriminator(nn.Module):
    def __init__(self, in_dim, out_dim=1):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(in_dim, 500)
        self.fc2 = nn.Linear(500, 500)
        self.fc3 = nn.Linear(500, out_dim)
        
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        
        x = self.fc3(x)
        x = F.sigmoid(x)
        
        return x

# Train/Eval Functions

In [14]:
def train_loop_classifier(source_feature_extractor, target_feature_extractor, discriminator,
                          source_loader, target_loader, loss_func, optimizer_discriminator, optimizer_target,
                          epoch, summary_loss_discriminator, summary_loss_target, device=None):
    source_feature_extractor.eval()
    target_feature_extractor.train()
    discriminator.train()
    
    for batch_idx, ((data, label), (data_target, _)) in enumerate(zip(source_loader, target_loader)):
        if device is not None:
            data, label, data_target = data.to(device), label.to(device), data_target.to(device)
    
        optimizer_discriminator.zero_grad()
        
        source_feature = source_feature_extractor(data)
        source_feature = torch.flatten(source_feature, 1)
        target_feature = target_feature_extractor(data_target)
        target_feature = torch.flatten(target_feature, 1)
        
        feature = torch.cat([source_feature, target_feature], dim=0)
        discriminator_output = discriminator(feature)
        label = torch.cat([torch.zeros((data.shape[0], 1)).to(device), torch.ones((data_target.shape[0], 1)).to(device)], dim=0)
        
        loss = loss_func(discriminator_output, label)
        loss.backward()
        optimizer_discriminator.step()
        
        optimizer_discriminator.zero_grad()
        optimizer_target.zero_grad()
        
        target_feature = target_feature_extractor(data_target)
        target_feature = torch.flatten(target_feature, 1)
        target_output = discriminator(target_feature)
        
        label = torch.zeros((data_target.shape[0], 1)).to(device)
        loss_target = loss_func(target_output, label)
        loss_target.backward()
        optimizer_target.step()
        
        summary_loss_discriminator.update(loss.detach().item(), BATCH_SIZE * 2)
        summary_loss_target.update(loss_target.detach().item(), BATCH_SIZE)

In [15]:
def eval_loop_classifier(target_feature_extractor, classifier,
                          target_loader, loss_func,
                          epoch, summary_loss, summary_acc_classifier=None, device=None):
    target_feature_extractor.eval()
    classifier.eval()
    
    with torch.no_grad():
        for batch_idx, (data_target, label_target) in enumerate(target_loader):
            if device is not None:
                data_target, label_target = data_target.to(device), label_target.to(device)

            target_feature = target_feature_extractor(data_target)
            target_feature = torch.flatten(target_feature, 1)
            output = classifier(target_feature)
            loss = loss_func(output, label_target)
            
            summary_loss.update(loss.detach().item(), BATCH_SIZE)
            if summary_acc_classifier is not None:
                acc = calculate_accuracy(output, label_target)
                summary_acc_classifier.update(acc, BATCH_SIZE)
    return summary_loss, summary_acc_classifier

In [16]:
def run_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    source_feature_extractor = FeatureExtractor()
    source_feature_extractor.load_state_dict(torch.load(os.path.join(PRETRAIN_DIR, "feature_extractor.pth")))
    
    target_feature_extractor = FeatureExtractor()
    target_feature_extractor.load_state_dict(torch.load(os.path.join(PRETRAIN_DIR, "feature_extractor.pth")))
    
    classifier = Classifier(400)
    classifier.load_state_dict(torch.load(os.path.join(PRETRAIN_DIR, "classifier.pth")))
    
    discriminator = Discriminator(400)
    
    source_feature_extractor.to(device)
    target_feature_extractor.to(device)
    discriminator.to(device)
    classifier.to(device)
    
    criterion_discriminator = nn.BCELoss()
    criterion_classifier = nn.CrossEntropyLoss()
    
    model_parameters_discriminator = list(discriminator.parameters())
    optimizer_discriminator = optim.Adam(model_parameters_discriminator, lr=LR, betas=(BETA1, BETA2))
    model_parameters_target = list(target_feature_extractor.parameters())
    optimizer_target = optim.Adam(model_parameters_target, lr=LR, betas=(BETA1, BETA2))
    
    source_train_loader = DataLoader(dataset=train_dataset_svhn, batch_size=BATCH_SIZE)
    target_train_loader = DataLoader(dataset=train_dataset_mnist, batch_size=BATCH_SIZE)
    
    source_valid_loader = DataLoader(dataset=valid_dataset_svhn, batch_size=BATCH_SIZE)
    target_valid_loader = DataLoader(dataset=valid_dataset_mnist, batch_size=BATCH_SIZE)

    best_epoch = None
    best_loss = None
    epoch = 0
    es_count = 0
    while(epoch < EPOCHS):
        epoch += 1
        summary_loss_discriminator = AverageMeter()
        summary_loss_target = AverageMeter()
        
        train_loop_classifier(source_feature_extractor, target_feature_extractor, discriminator,
                              source_train_loader, target_train_loader, criterion_discriminator, optimizer_discriminator, optimizer_target,
                              epoch-1, summary_loss_discriminator, summary_loss_target, device=device)

        summary_loss_valid = AverageMeter()
        summary_acc_valid = AverageMeter()

        eval_loop_classifier(target_feature_extractor, classifier,
                             target_valid_loader, criterion_classifier,
                             epoch-1, summary_loss_valid, summary_acc_valid, device=device)

        print(f"#### EPOCH {epoch}/{EPOCHS} ####")
        print(f"[iterations]{len(summary_loss_discriminator) * epoch} [train discriminator loss]{summary_loss_discriminator.avg:.3f} [train target loss]{summary_loss_target.avg:.3f}")
        print(f"[valid loss]{summary_loss_valid.avg:.3f} [valid acc]{summary_acc_valid.avg:.3f}")
        print(f"#######################")
        
        if best_loss is None:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch

        if best_loss > summary_loss_valid.avg:
            best_loss = summary_loss_valid.avg
            best_epoch = epoch
            es_count = 0
        else:
            es_count += 1

    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [17]:
best_epoch = run_training()

#### EPOCH 1/100 ####
[iterations]422 [train discriminator loss]0.079 [train target loss]3.876
[valid loss]2.399 [valid acc]0.111
#######################
#### EPOCH 2/100 ####
[iterations]844 [train discriminator loss]0.001 [train target loss]6.200
[valid loss]2.416 [valid acc]0.109
#######################
#### EPOCH 3/100 ####
[iterations]1266 [train discriminator loss]0.006 [train target loss]9.125
[valid loss]4.402 [valid acc]0.225
#######################
#### EPOCH 4/100 ####
[iterations]1688 [train discriminator loss]0.058 [train target loss]17.202
[valid loss]1.874 [valid acc]0.369
#######################
#### EPOCH 5/100 ####
[iterations]2110 [train discriminator loss]0.029 [train target loss]5.573
[valid loss]1.827 [valid acc]0.383
#######################
#### EPOCH 6/100 ####
[iterations]2532 [train discriminator loss]0.020 [train target loss]6.290
[valid loss]2.140 [valid acc]0.343
#######################
#### EPOCH 7/100 ####
[iterations]2954 [train discriminator loss]0.018 

In [18]:
def run_second_training():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    source_feature_extractor = FeatureExtractor()
    source_feature_extractor.load_state_dict(torch.load(os.path.join(PRETRAIN_DIR, "feature_extractor.pth")))
    
    target_feature_extractor = FeatureExtractor()
    target_feature_extractor.load_state_dict(torch.load(os.path.join(PRETRAIN_DIR, "feature_extractor.pth")))
    
    discriminator = Discriminator(400)
    
    source_feature_extractor.to(device)
    target_feature_extractor.to(device)
    discriminator.to(device)
    
    criterion_discriminator = nn.BCELoss()
    criterion_classifier = nn.CrossEntropyLoss()
    
    model_parameters_discriminator = list(discriminator.parameters())
    optimizer_discriminator = optim.Adam(model_parameters_discriminator, lr=LR, betas=(BETA1, BETA2))
    model_parameters_target = list(target_feature_extractor.parameters())
    optimizer_target = optim.Adam(model_parameters_target, lr=LR, betas=(BETA1, BETA2))
    
    source_train_loader = DataLoader(dataset=train_valid_dataset_svhn, batch_size=BATCH_SIZE)
    target_train_loader = DataLoader(dataset=train_valid_dataset_mnist, batch_size=BATCH_SIZE)

    epoch = 0
    es_count = 0
    while(epoch < best_epoch):
        epoch += 1
        summary_loss_discriminator = AverageMeter()
        summary_loss_target = AverageMeter()
        
        train_loop_classifier(source_feature_extractor, target_feature_extractor, discriminator,
                              source_train_loader, target_train_loader, criterion_discriminator, optimizer_discriminator, optimizer_target,
                              epoch-1, summary_loss_discriminator, summary_loss_target, device=device)

        
        print(f"#### EPOCH {epoch}/{EPOCHS} ####")
        print(f"[iterations]{len(summary_loss_discriminator) * epoch} [train discriminator loss]{summary_loss_discriminator.avg:.3f} [train target loss]{summary_loss_target.avg:.3f}")
        print(f"#######################")

    print(f"Best epoch: {best_epoch}")
    return target_feature_extractor

In [19]:
target_feature_extractor = run_second_training()

#### EPOCH 1/100 ####
[iterations]469 [train discriminator loss]0.079 [train target loss]3.756
#######################
#### EPOCH 2/100 ####
[iterations]938 [train discriminator loss]0.001 [train target loss]6.347
#######################
#### EPOCH 3/100 ####
[iterations]1407 [train discriminator loss]0.000 [train target loss]7.663
#######################
#### EPOCH 4/100 ####
[iterations]1876 [train discriminator loss]0.005 [train target loss]14.172
#######################
#### EPOCH 5/100 ####
[iterations]2345 [train discriminator loss]0.000 [train target loss]27.630
#######################
#### EPOCH 6/100 ####
[iterations]2814 [train discriminator loss]0.076 [train target loss]11.675
#######################
#### EPOCH 7/100 ####
[iterations]3283 [train discriminator loss]0.031 [train target loss]5.863
#######################
#### EPOCH 8/100 ####
[iterations]3752 [train discriminator loss]0.028 [train target loss]6.874
#######################
#### EPOCH 9/100 ####
[iterations]4221 

In [20]:
def run_test(target_feature_extractor):
    classifier = Classifier(400)
    classifier.load_state_dict(torch.load(os.path.join(PRETRAIN_DIR, "classifier.pth")))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    target_feature_extractor.to(device)
    classifier.to(device)
    
    target_feature_extractor.eval()
    classifier.eval()
    
    test_loader = DataLoader(dataset=test_dataset_mnist, 
                             batch_size=BATCH_SIZE,
                             shuffle=True)
    
    criterion = nn.CrossEntropyLoss()
    
    summary_loss_test = AverageMeter()
    summary_acc_test = AverageMeter()

    summary_loss_test, summary_acc_test = eval_loop_classifier(target_feature_extractor, classifier, test_loader, 
                                               criterion, 1, summary_loss_test, summary_acc_test, device=device)

    print(f"[test loss]{summary_loss_test.avg} [test acc]{summary_acc_test.avg}")

In [21]:
run_test(target_feature_extractor)

[test loss]1.9365125426763221 [test acc]0.36965981125831604
